INFO 08-12 10:27:11 [config.py:3440] Downcasting torch.float32 to torch.float16.
INFO 08-12 10:27:12 [config.py:1604] Using max model len 128
INFO 08-12 10:27:12 [config.py:4628] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
INFO 08-12 10:27:12 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', speculative_config=None, tokenizer='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=128, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitesp

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00, 53.03it/s]



INFO 08-12 10:27:17 [default_loader.py:262] Loading weights took 0.27 seconds
INFO 08-12 10:27:19 [model_runner.py:1115] Model loading took 0.2215 GiB and 0.687240 seconds


Processed prompts: 100%|██████████| 2/2 [00:00<00:00, 135.12it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [ ]:
from llama_cpp import Llama
import json
from pydantic import BaseModel, Field
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from vllm import LLM
import pandas as pd
import re
from time import time

def extrair_json_do_markdown(texto: str):
    # Extrai o bloco JSON
    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if not match:
        raise ValueError("Não foi encontrado JSON válido no texto")
    json_str = match.group(0)

    # Corrige aspas duplas internas que não estejam escapadas
    # Substitui aspas duplas repetidas por aspas simples (ou remove extras)
    # Cuidado: essa regex pode precisar ser ajustada conforme o padrão do seu texto
    json_str = re.sub(r'""', '"', json_str)

    # Também pode tentar escapar aspas internas entre aspas:
    # json_str = re.sub(r'(?<=: )"([^"]*)"', lambda m: '"' + m.group(1).replace('"', '\\"') + '"', json_str)

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Erro ao decodificar JSON: {e}\nJSON extraído:\n{json_str}")



class EmergencyInterpretationA(BaseModel):
    descricao_breve: str = Field(description="Breve descrição da ocorrência da chamada de emergência (máximo de 200 caracteres).")
    outras_observacoes: str = Field(description="Outras observações que o solicitante tenha feito durante a transcrição, mas que não estão presentes na descrição breve.")
    #tipo_chamada: str = Field(description="Tipo de ligação recebida.",
    #    enum=["Ocorrência", 'Ligação Muda', 'Trote', 'Queda de Ligação', 'Informação', 'Agradecimento', 'Denúncia'])

emergencyInterpretationA_schema_str = json.dumps(
    EmergencyInterpretationA.model_json_schema(), 
    ensure_ascii=False, indent=2)


class NaturezaOcorrenciaA(BaseModel):
    natureza_da_ocorrencia_indice: int = Field(
        description="Indice da natureza de ocorrência mais adequada. Usar 0 (zero) quando não for nenhuma das opções.",
        default=0)

emergencyInterpretationB_schema_str = json.dumps(
    NaturezaOcorrenciaA.model_json_schema(), 
    ensure_ascii=False, indent=2)

prompt_a_template = """
You are an expert in interpreting call transcripts for emergency services. Your task is to extract relevant information
from the provided call transcript.
Please analyze the following call transcript and extract the required information:
{[[[[transcript_content]]]]}
"""

prompt_b_template = """
Você receberá a transcrição de uma chamada de emergência de ocorrência. Deve decidir qual, 
dentre uma série de naturezas de ocorrência padronizadas, é a mais adequada. 
Cada natureza tem um índice e você deverá responder com o indice da opção correta.
Responda apenas com o índice e nada mais.
Usar 0 (zero) quando não for nenhuma das opções.

Lista de naturezas:
\nnaturezas_similares

\nTranscrição da chamada:
[[[[transcript_content]]]]
"""

user_template_with_schema = f"""
Você é um assistente especializado em extrair informações de transcrições de chamadas de emergência.
{emergencyInterpretationA_schema_str}
Sua tarefa é analisar a transcrição abaixo e produzir um JSON estritamente válido de acordo com o seguinte schema:
[[[[transcript_content]]]]
"""

naturezas_path = 'naturezas.csv'
naturezas_vec = pd.read_csv(naturezas_path, sep=';')['NO_NATUREZA_INICIAL'].to_list()

emergencia1 = """Operador: Bom dia, precisamos verificar o que está acontecendo. Por favor, 
    descreva o que está ocorrendo.\n\nSolicitante: Deus... tem um fogo! No prédio!\n\nOperador: 
    Onde exatamente? Pode me dar o endereço?\n\nSolicitante: Praça Antônio Carlos, número seis. 
    A construção da Lorrana Negretti.\n\nOperador: Entendo. É um incêndio em um edifício 
    residencial. Você pode me dizer o que está acontecendo? As pessoas estão em perigo?\n\n
    Solicitante: Eu só cheguei há pouco tempo, vi fumaça, fogo alto... as pessoas gritando. 
    Não sei o que está pegando.\n\nOperador: Tente me dar mais detalhes. Há pessoas presas? 
    Você consegue ver o fogo de onde você está?\n\nSolicitante: Está muito forte, a fumaça 
    é muito grossa. Eu estou na praça, vendo as chamas. As pessoas estão saindo apressadas.
    \n\nOperador: Ok. Precisamos avaliar a situação. Você consegue ver se há algum ferido ou 
    pessoas em perigo imediato?\n\nSolicitante: Não consigo ver bem. A fumaça está me sufocando. 
    Apenas vejo as chamas e as pessoas correndo.\n\nOperador: Tente respirar fundo e me fale 
    se você consegue ver alguma coisa que possa ajudar a identificar a área mais afetada. 
    Há algum cheiro específico?\n\nSolicitante: Cheira muito a fumaça, queimado. Eu não sei o 
    que mais.\n\nOperador: Estamos enviando ajuda. Por favor, permaneça onde você está, a 
    menos que seja seguro fazê-lo. Você consegue me dar seu nome, por favor?\n\nSolicitante: 
    Arthur... Arthur Gabriel Porto.\n\nOperador: Ok, Arthur. Ajudamos você, Arthur. A equipe 
    de emergência já está a caminho. Fique calmo e siga as instruções da equipe quando chegarem."""

emergencia2 = """Operador: Bom dia, emergência? Por favor, explique o que está acontecendo.\n\nS
olicitante: (Gaguejando) A... a ajuda! Por favor, preciso de ajuda! Estou sendo atacado!\n\n
Operador: Calma, senhor. Onde você está? Pode me falar seu endereço?\n\nSolicitante: Rua... Rua 
Magnólia, número vinte. Ali, na Serra do Salitre. Tem um... um animal me atacando!\n\nOperador: Ok, 
Rua Magnólia, 20, Serra do Salitre. Pode me descrever o que está acontecendo?\n\nSolicitante: (Em 
pânico) É um cachorro! Um cachorro grande... ele me mordeu! Estou com medo!\n\nOperador: Senhor, 
mantenha a calma. Onde exatamente você está sendo atacado? Há pessoas por perto?\n\nSolicitante: 
(Chorando) Está acontecendo aqui, na rua... Não consigo me mover! Ele continua me mordendo!\n\nOperador: 
Entendo. A situação é grave. Você consegue me dar o seu nome, por favor?\n\nSolicitante: Asafe... 
Asafe Lopes.\n\nOperador: Sr. Asafe Lopes. Preciso de mais informações para acionar o apoio adequado. 
Você consegue me dizer o que aconteceu?\n\nSolicitante: (Ainda em pânico) Um cachorro... ele me pegou 
de surpresa! Eu não sei o que aconteceu... estou com dor!\n\nOperador: Ok, Sr. Asafe, tente se acalmar. 
Estamos enviando ajuda. Você consegue me dizer algo mais sobre o animal? Ele é perigoso?\n\nSolicitante: 
(Tremendo) Sim! É enorme! Ele não está parando!\n\nOperador: Tudo bem, Sr. Asafe. A ajuda já está a 
caminho. Fique onde está e não se mova, ok?\n\nSolicitante: (Soluçando) Ok... ok...\n\nOperador: Fique 
tranquilo, Sr. Asafe. Estamos monitorando a situação.\n"""

class LlamaCppRunner:
    def __init__(self, repo_id="cnmoro/Gemma-3-Gaia-PT-BR-4b-it-Q8_0-GGUF",
            filename="gemma-3-gaia-pt-br-4b-it-q8_0.gguf",
            low_end_gpu = True):
        self.repo_id = repo_id
        self.filename = filename
        if low_end_gpu:
            self.llm = Llama.from_pretrained(
                repo_id=repo_id,
                filename=filename,
                n_gpu_layers=24,     # 0 = só CPU; use -1 para tudo na GPU (se houver VRAM),
                max_tokens=2048,
                verbose=False,
                rope_scaling_type=1,   # Enable RoPE scaling
                n_ctx=1200  # Defina o tamanho do contexto explicitamente
            )
        else:
            self.llm = Llama.from_pretrained(
                repo_id=repo_id,
                filename=filename,
                n_gpu_layers=-1,     # 0 = só CPU; use -1 para tudo na GPU (se houver VRAM),
                max_tokens=2048,
                verbose=False,
                rope_scaling_type=1,   # Enable RoPE scaling
                n_ctx=2400  # Defina o tamanho do contexto explicitamente
            )
    
    def create_interpretation_a(self, transcription: str):
        user_prompt = user_template_with_schema.replace('[[[[transcript_content]]]]', transcription)
        s = time()
        resp = self.llm.create_chat_completion(
            messages=[
                {"role": "system", "content": "Você é um assistente que sempre responde estritamente no formato JSON especificado."},
                {"role": "user", "content": user_prompt}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": emergencyInterpretationA_schema_str
            },
        )
        interval = time() - s

        try:
            resp2 = extrair_json_do_markdown(resp['choices'][0]['message']['content'])
        except Exception as err:
            print(err)
            return resp, err
        return resp2, {'meta': {'processing_time': interval}}
    
    def create_interpretation_b(self, transcription: str, classifications: list):
        classifications_indexed = {n+1: c for n, c in enumerate([x for s, x in classifications])}
        classifications_json = json.dumps(classifications_indexed, ensure_ascii=False)
        print(classifications_json)
        user_prompt = prompt_b_template.replace(
            'naturezas_similares', classifications_json).replace(
                '[[[[transcript_content]]]]', transcription
            )
        
        s = time()
        resp = self.llm.create_chat_completion(
            messages=[
                {"role": "system", "content": "Você é um assistente que sempre responde estritamente no formato especificado."},
                {"role": "user", "content": user_prompt}
            ]
        )
        interval = time()-s

        nat_id = resp['choices'][0]['message']['content']
        try:
            nat_id = int(nat_id)
        except Exception as err:
            print(err)
            return resp, {'meta': {'message': resp['choices'][0]['message'],
                'error': err}}
        print('natureza_correta', nat_id)
        if nat_id in classifications_indexed:
            nat_name = classifications_indexed[nat_id]
        else:
            nat_name = None
        print(nat_name)
        return nat_name, {'meta': {'message': resp['choices'][0]['message'], 
            'processing_time': interval}
            }

class VllmEmbeddingStore:
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.model_name = model_name
        self.embedding_llm = LLM(model=model_name, task="embed")
        self.embs = []
        self.docs = []
        self.n_tokens = []

    def add_documents(self, docs):
        outputs = self.embedding_llm.embed(docs)
        embs = [np.array(x.outputs.embedding) for x in outputs]
        n_tokens = [len(x.prompt_token_ids) for x in outputs]
        self.embs += embs
        self.docs += docs
        self.n_tokens += n_tokens

    def search(self, new_docs):
        new_outputs = self.embedding_llm.embed(new_docs)
        new_embs = np.asarray([np.array(x.outputs.embedding) for x in new_outputs])
        new_n_tokens = [len(x.prompt_token_ids) for x in new_outputs]
        sims = cosine_similarity(np.asarray(self.embs), new_embs)
        sims_dict = {d: [] for d in new_docs}
        for doc, docs_sims in zip(self.docs, sims):
            for new_doc, score in zip(new_docs, docs_sims):
                sims_dict[new_doc].append([score, doc])
        
        for d, scores in sims_dict.items():
            scores.sort(reverse=True)

        return sims_dict

class EmergencyInterpreter():
    SUPPORTED_INTERPRETERS = ['llama_cpp']
    def __init__(self, naturezas_vec, interpreter_type="llama_cpp"):
        assert interpreter_type in EmergencyInterpreter.SUPPORTED_INTERPRETERS
        if interpreter_type=="llama_cpp":
            self.interpreter = LlamaCppRunner()
        self.embedding_store = VllmEmbeddingStore()
        self.embedding_store.add_documents(naturezas_vec)
    
    def process_emergencies(self, contexts, max_naturezas=10):
        transcriptions = [x['transcription'] for x in contexts]
        interpretations_a = [self.interpreter.create_interpretation_a(t) 
            for t in transcriptions]
        meta_dicts_a = [m for _, m in interpretations_a]
        interpretations_a = [x for x, m in interpretations_a]
        
        descs = [r['descricao_breve'] for r in interpretations_a]
        similares = self.embedding_store.search(descs)
        for desc in descs:
            similares[desc] = similares[desc][:max_naturezas]
        
        similares_vecs = [similares[desc] for desc in descs]
        interpretations_b = [self.interpreter.create_interpretation_b(e, s)
            for e, s in zip(transcriptions, similares_vecs)]
        meta_dicts_b = [m for _, m in interpretations_b]
        interpretations_b = [x for x, m in interpretations_b]

        results_final = []
        for n, transcription in enumerate(transcriptions):
            desc = descs[n]
            results_final.append({
                "classificacoes_provaveis": similares[desc],
                "classificacao_decisiva": interpretations_b[n],
                "descricao_breve": desc,
                "outras_observacoes": interpretations_a[n]['outras_observacoes'],
                "transcricao": transcription,
                "meta_a": meta_dicts_a[n],
                "meta_b": meta_dicts_b[n]
            })
        
        return results_final

complete_interpreter = EmergencyInterpreter(naturezas_vec)

INFO 08-12 16:55:03 [__init__.py:235] Automatically detected platform cuda.


llama_context: n_ctx_per_seq (1200) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


INFO 08-12 16:55:09 [config.py:642] Found sentence-transformers tokenize configuration.
INFO 08-12 16:55:23 [config.py:538] Found sentence-transformers modules configuration.
INFO 08-12 16:55:23 [config.py:558] Found pooling configuration.
INFO 08-12 16:55:24 [config.py:3440] Downcasting torch.float32 to torch.float16.
INFO 08-12 16:55:24 [config.py:1604] Using max model len 128
WARNING 08-12 16:55:24 [arg_utils.py:1690] ['BertModel', 'TransformersForCausalLM'] is not supported by the V1 Engine. Falling back to V0. 
INFO 08-12 16:55:24 [config.py:4628] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
INFO 08-12 16:55:24 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', speculative_config=None, tokenizer='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_r

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00, 61.72it/s]



INFO 08-12 16:55:31 [default_loader.py:262] Loading weights took 0.43 seconds
INFO 08-12 16:55:32 [model_runner.py:1115] Model loading took 0.2207 GiB and 1.525247 seconds


Processed prompts: 100%|██████████| 591/591 [00:00<00:00, 1020.48it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [2]:
contexts = [{'transcription': emergencia1}, {'transcription': emergencia2}]
responses = complete_interpreter.process_emergencies(contexts)

queries_simples = [
    'Prédio prestes a desabar',
    'Pedro está quase se afogando na praia',
    'O prédio está pegando fogo',
    'Incêndio',
    "Vizinho ouviu Carlos agredindo sua esposa Maria",
    "Pedro teve seu carro roubado enquanto abria a garagem.",
    "Patricia acourdou, foi até a cozinha, viu ela pegando fogo e saiu correndo do prédio",
]

new_contexts = [{'transcription': x} for x in queries_simples]
resposes2 = complete_interpreter.process_emergencies(new_contexts)


Processed prompts: 100%|██████████| 2/2 [00:00<00:00, 144.89it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


{"1": "Incêndios Em Aglomerados Residenciais", "2": "Remoção de Cadáver Vítima de Incêndio", "3": "Incêndio Em Residência", "4": "Perícia Em Local de Incêndio", "5": "Incêndio Em Outros", "6": "Incêndio Em Rede Elétrica", "7": "Perícia de Incêndio", "8": "Incêndio Em Edificação", "9": "Morte Acidental Provocada Por Queimaduras", "10": "Incêndio Florestal"}
natureza_correta 3
Incêndio Em Residência
{"1": "Ataque de Animal", "2": "Traumático/acidente Com Animal Peçonhento", "3": "Animal Em Situação de Risco", "4": "Emergência Psiquiátrica", "5": "Maltrato de Animais", "6": "Mordida de Animal", "7": "Animal Em Via Pública", "8": "Atropelamento de Animal", "9": "Captura/resgate de Animal", "10": "Infestações de Animais"}
natureza_correta 6
Mordida de Animal


Processed prompts: 100%|██████████| 7/7 [00:00<00:00, 480.30it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


{"1": "Remoção de Cadáver Por Espancamento", "2": "Obstrução de Esgoto", "3": "Atropelamento", "4": "Abandono de Incapaz", "5": "Deslizamentos", "6": "Furto À Residência", "7": "Fuga de Preso", "8": "Rompimento/ Colapso de Barragens", "9": "Remoção de Cadáver Por Sufocamento Ou Esganadura", "10": "Fuga de Menor"}
natureza_correta 5
Deslizamentos
{"1": "Ressaca Marítima", "2": "Colocação de Adriça", "3": "Naufrágio", "4": "Pessoa Desaparecida", "5": "Onda de Calor", "6": "Remoção de Cadáver Vítima de Queda", "7": "Apoio Ao Samu", "8": "Levantamento Papiloscópico", "9": "Tempestade de Raios", "10": "Vítima Ejetada de Veículo"}
natureza_correta 3
Naufrágio
{"1": "Perícia de Incêndio", "2": "Incêndio Em Residência", "3": "Incêndio Em Edificação", "4": "Perícia Em Local de Incêndio", "5": "Incêndio Em Rede Elétrica", "6": "Incêndio Em Outros", "7": "Incêndio Em Lixo", "8": "Remoção de Cadáver Vítima de Incêndio", "9": "Queimadura Térmica", "10": "Incêndio Em Vegetação"}
natureza_correta 2
I

In [7]:
all_responses = responses + resposes2
for r in all_responses:
    print('Transcrição:')
    if len(r['transcricao']) > 100:
        print(r['transcricao'][:100].replace('\n',' '))
    else:
        print(r['transcricao'].replace('\n',' '))
    print('\tdescricao breve', r['descricao_breve'])
    print('\tclassificacao decisiva', r['classificacao_decisiva'])
    print('\toutras observacoes', r['outras_observacoes'])
    print('\tT1', r['meta_a']['meta']['processing_time'])
    print('\tT2', r['meta_b']['meta']['processing_time'])

Transcrição:
Operador: Bom dia, precisamos verificar o que está acontecendo. Por favor,      descreva o que está 
	descricao breve Incêndio em um edifício residencial na Praça Antônio Carlos, número seis, construção da Lorrana Negretti. Pessoas em perigo e chamas visíveis.
	classificacao decisiva Incêndio Em Residência
	outras observacoes Solicitante relata fumaça densa, pessoas gritando e chamas fortes. Não consegue identificar a causa ou feridos.
	T1 10.823472023010254
	T2 1.2148258686065674
Transcrição:
Operador: Bom dia, emergência? Por favor, explique o que está acontecendo.  S olicitante: (Gaguejand
	descricao breve Homem relatando ataque de cachorro na Rua Magnólia, 20, Serra do Salitre. Solicita ajuda e informa sobre mordidas e pânico.
	classificacao decisiva Mordida de Animal
	outras observacoes O solicitante demonstra pânico e medo, com dificuldade em articular detalhes além da descrição do ataque.
	T1 9.292720556259155
	T2 1.1192851066589355
Transcrição:
Prédio prestes a des

In [6]:
responses[0]

{'classificacoes_provaveis': [[np.float64(0.8275916316689185),
   'Incêndios Em Aglomerados Residenciais'],
  [np.float64(0.7669572966467683), 'Remoção de Cadáver Vítima de Incêndio'],
  [np.float64(0.7325240016308421), 'Incêndio Em Residência'],
  [np.float64(0.7068710297651323), 'Perícia Em Local de Incêndio'],
  [np.float64(0.7017959374978053), 'Incêndio Em Outros'],
  [np.float64(0.6940626785691054), 'Incêndio Em Rede Elétrica'],
  [np.float64(0.692551033291491), 'Perícia de Incêndio'],
  [np.float64(0.671894651264958), 'Incêndio Em Edificação'],
  [np.float64(0.65083400131254), 'Morte Acidental Provocada Por Queimaduras'],
  [np.float64(0.6479800654612048), 'Incêndio Florestal']],
 'classificacao_decisiva': 'Incêndio Em Residência',
 'descricao_breve': 'Incêndio em um edifício residencial na Praça Antônio Carlos, número seis, construção da Lorrana Negretti. Pessoas em perigo e chamas visíveis.',
 'outras_observacoes': 'Solicitante relata fumaça densa, pessoas gritando e chamas for